# 实验 2：$G$ 能否追踪离开 0.5 的未知边界？

研究问题只有一句：

> 当动力学把未知边界 $\kappa^*$ 推离 $0.5$ 时，$G(t,\kappa)$ 的峰能否追踪它？

实验只改变双记忆的权重：

$$W_\delta=\frac{(1+\delta)AA^\top+(1-\delta)BB^\top}{N},\qquad
\delta\in\{-0.30,-0.15,0,0.15,0.30\}.$$

查询线、连续动力学、`jacfwd` 拉回度量和不读取 $G$ 的长时二分，都沿用实验 1。

## 1. 冻结源码

Notebook 固定到已测试的源码提交，并分别核对实验 1 公共核心和实验 2 源码哈希。

In [ ]:
import hashlib
import importlib.util
from pathlib import Path
import subprocess
import sys
import urllib.request

required = {
    "jax": "jax[cpu]",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

CODE_REV = "ffca0ad59857e19e24d6bab3746d73265bdf4237"
BASE = Path("representation-geometry/experiments/hopfield-dynamic-geometry")
SOURCES = {
    "experiment_01_boundary_localization.py": "c40bf773180ccd7ebced4c9413851878ae868e27b0c7207b3d1ff1abd0f79ec2",
    "experiment_02_moving_boundary.py": "58bd4699e94bfc11b83678b1b305280008269d42977b2d67ecfc41c5420da86b",
}

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

local_directory = Path.cwd() / BASE
if all((local_directory / name).is_file() and sha256(local_directory / name) == digest
       for name, digest in SOURCES.items()):
    source_directory = local_directory
else:
    source_directory = Path("/content/hopfield_dynamic_geometry")
    source_directory.mkdir(parents=True, exist_ok=True)
    for name, digest in SOURCES.items():
        path = source_directory / name
        url = (
            "https://raw.githubusercontent.com/Heptazero/nn-labs/"
            f"{CODE_REV}/{BASE.as_posix()}/{name}"
        )
        urllib.request.urlretrieve(url, path)
        if sha256(path) != digest:
            raise RuntimeError(f"源码哈希不匹配: {name}")

sys.path.insert(0, str(source_directory))
print("source:", source_directory)
print("revision:", CODE_REV)

## 2. 冻结条件与判定规则

- $N=60$，8 个 seed，A/B overlap 固定为 0，只扫描 $\delta$；
- 终点必须收敛，A、B 必须仍落入不同吸引子；
- 长时标签沿查询线必须只切换一次，否则记为 `topology_changed`；
- 真实边界由长时 overlap margin 二分，计算过程不能读取 $G$；
- 同一批轨迹比较 $G$ 峰、当前 overlap 零点、当前查询曲线的连续 Hopfield 能量 ridge。

`passed_boundary_tracking` 只回答 $G$ 能否追踪边界。`supports_independent_G_value` 还要求 $G$ 比两个简单基线更早或更准。

In [ ]:
from experiment_02_moving_boundary import (
    MovingBoundaryConfig,
    run_experiment,
    write_artifacts,
)

config = MovingBoundaryConfig()
config

## 3. 运行正式扫描

第一次运行需要编译 JAX。CPU 上通常约一分钟。所有估计先用 float64 计算；密集原始网格仅在写盘时压成 float32。

In [ ]:
from IPython.display import display
import pandas as pd

conditions, time_results, grids = run_experiment(config)
output_dir = (
    Path("/content/experiment_02_outputs")
    if Path("/content").exists()
    else Path("/tmp/nn_labs_experiment_02_outputs")
)
summary, conclusion_path = write_artifacts(
    output_dir, conditions, time_results, grids, config
)

print("output:", output_dir)
display(pd.DataFrame([summary]))
display(
    conditions.groupby("delta", as_index=False).agg(
        boundary_kappa=("boundary_kappa", "median"),
        valid_topology=("topology_status", lambda values: (values == "valid").mean()),
    )
)

## 4. 主图

A 检查边界是否真的移动。B 显示代表条件中三个估计器随时间的位置。C 在所有非零 $\delta$ 上直接比较误差。D 把 $G$ 的峰轨迹叠在度量热图上。

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(output_dir / "main_figure.png")))

## 5. 本次冻结结果怎样解释

正式产物得到：

- 五个 $\delta$ 都保留两个稳定终点和一次边界切换；
- $\kappa^*$ 依次约为 $0.443,0.473,0.500,0.527,0.557$；
- $t\ge2$ 时，$G$ 的中位误差约 $0.00116$，90% 分位约 $0.00167$，所以**边界追踪校准通过**；
- 在非零 $\delta$ 上，晚期中位误差为：overlap 零点 $0.00112$，$G$ 峰 $0.00128$，能量 ridge $0.00553$；
- 持续进入 $0.01$ 误差带的中位时刻为：能量 ridge $0$，overlap 零点 $0.55$，$G$ 峰 $0.75$。

因此第二层是负结果：$G$ 能追踪真正移动的边界，但既没有比能量更早，也没有比 overlap 零点更准。本实验暂不支持 $G$ 的独立分析价值；它仍是一种清楚的动态几何可视化。

## 6. 原始产物

输出包括配置、40 个条件的拓扑与终点检查、逐时刻三个估计器、估计器汇总、完整观测量网格、主图和结论草稿。

In [ ]:
import shutil

archive = shutil.make_archive(str(output_dir), "zip", root_dir=output_dir)
print("archive:", archive)
try:
    from google.colab import files
    print("在 Colab 中运行 files.download(archive) 即可下载。")
except ImportError:
    pass